# SO-101 cube pick-and-place · ACT on Colab L4
Select an NVIDIA L4 GPU runtime and enable notebook access for `github_token` and `hf_token`. Run the smoke test before the main run. The policy learns a single task: place the red cube in the blue box.

In [ ]:
import subprocess
import sys

if sys.version_info < (3, 12):
    raise RuntimeError("Select a Colab runtime with Python 3.12 or newer.")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "lerobot[dataset,training]==0.6.1"],
    check=True,
)

In [ ]:
import math
import shutil
from pathlib import Path

import pyarrow.parquet as pq
import requests
import torch
from google.colab import drive, userdata
from huggingface_hub import snapshot_download
from lerobot.datasets import LeRobotDataset
from lerobot.datasets.compute_stats import aggregate_stats
from lerobot.datasets.io_utils import cast_stats_to_numpy, write_stats
from lerobot.utils.utils import unflatten_dict

PROJECT = "Jay-Aditya-16/so101-grasp-recovery"
DATASET = "caspar/so101_pickplace_sim_150"
REVISION = "b4fb342927bb0ee7dcbe4b90472a0c257ce4f5de"
TASK = "Pick up the red cube and put it in the blue box."
SEED = 1000
VALIDATION_FRACTION = 0.10

if not torch.cuda.is_available() or "L4" not in torch.cuda.get_device_name(0):
    raise RuntimeError("Select an NVIDIA L4 GPU in Colab.")

response = requests.get(
    f"https://api.github.com/repos/{PROJECT}",
    headers={"Authorization": f"Bearer {userdata.get('github_token')}"},
    timeout=20,
)
response.raise_for_status()
if not response.json()["private"]:
    raise RuntimeError("Expected the project repository to be private.")
del response

source_root = Path(snapshot_download(
    DATASET, repo_type="dataset", revision=REVISION,
    token=userdata.get("hf_token"), local_dir="/content/so101-dataset",
))
dataset = LeRobotDataset(DATASET, root=source_root, video_backend="pyav")
if (dataset.num_episodes, len(dataset)) != (117, 20124):
    raise RuntimeError("Unexpected dataset size.")
for index in (0, len(dataset) // 2, len(dataset) - 1):
    sample = dataset[index]
    if (
        sample["task"] != TASK
        or tuple(sample["observation.state"].shape) != (6,)
        or tuple(sample["action"].shape) != (6,)
        or any(
            tuple(sample[f"observation.images.{camera}"].shape) != (3, 240, 320)
            for camera in ("top", "wrist")
        )
    ):
        raise RuntimeError(f"Dataset contract mismatch at frame {index}.")
print(f"{len(dataset):,} frames across {dataset.num_episodes} episodes validated by metadata and sample reads.")
# Match LeRobot 0.6.1's per-task tail split for this single-task dataset.
n_validation = math.ceil(dataset.num_episodes * VALIDATION_FRACTION)
n_train = dataset.num_episodes - n_validation
metadata = pq.read_table(source_root / "meta" / "episodes").to_pylist()
training_rows = sorted(metadata, key=lambda row: row["episode_index"])[:n_train]
if [row["episode_index"] for row in training_rows] != list(range(n_train)):
    raise RuntimeError("Episode metadata is incomplete.")
episode_stats = [
    cast_stats_to_numpy(unflatten_dict({
        key.removeprefix("stats/"): value
        for key, value in row.items()
        if key.startswith("stats/") and value is not None
    }))
    for row in training_rows
]
if not all({"action", "observation.state"} <= stats.keys() for stats in episode_stats):
    raise RuntimeError("Per-episode statistics are missing.")

# Keep the downloaded revision intact and reuse its videos without copying them.
root = Path("/content/so101-training-dataset")
shutil.copytree(source_root / "meta", root / "meta", dirs_exist_ok=True)
for name in ("data", "videos"):
    if not (root / name).is_symlink():
        (root / name).symlink_to(source_root / name, target_is_directory=True)
write_stats(aggregate_stats(episode_stats), root)
print(f"Training episodes: 0–{n_train - 1}. Offline validation: {n_train}–{dataset.num_episodes - 1}.")
drive.mount("/content/drive")

In [ ]:
import zipfile
from datetime import datetime, timezone

TRAINER = shutil.which("lerobot-train")
if not TRAINER:
    raise RuntimeError("lerobot-train is missing. Run the installation cell.")
DRIVE_OUTPUT = Path("/content/drive/MyDrive/so101-act")
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 8  # If the smoke test runs out of memory, set to 4 and rerun it.

def train_act(label, steps, save_freq):
    run = f"{label}-{datetime.now(timezone.utc):%Y%m%dT%H%M%S}"
    output = Path("/content/so101-training") / run
    command = [
        TRAINER,
        f"--dataset.repo_id={DATASET}",
        f"--dataset.root={root}",
        f"--dataset.revision={REVISION}",
        "--dataset.video_backend=pyav",
        "--policy.type=act",
        "--policy.device=cuda",
        "--policy.push_to_hub=false",
        "--policy.chunk_size=50",
        "--policy.n_action_steps=20",
        f"--batch_size={BATCH_SIZE}",
        f"--dataset.eval_split={VALIDATION_FRACTION}",
        f"--eval_steps={save_freq}",
        f"--log_freq={min(200, steps)}",
        "--num_workers=2",
        f"--steps={steps}",
        f"--save_freq={save_freq}",
        "--env_eval_freq=0",
        "--wandb.enable=false",
        f"--seed={SEED}",
        f"--output_dir={output}",
    ]
    run_dir = DRIVE_OUTPUT / run
    run_dir.mkdir()
    exported = {}

    def export_latest():
        # LeRobot updates this symlink only after it finishes writing a checkpoint.
        try:
            checkpoint = (output / "checkpoints" / "last").resolve(strict=True)
        except FileNotFoundError:
            return
        if checkpoint.name in exported:
            return
        archive = run_dir / f"{checkpoint.name}.zip"
        partial = archive.with_suffix(".part")
        with zipfile.ZipFile(partial, "w", compression=zipfile.ZIP_STORED) as bundle:
            for file in sorted(checkpoint.rglob("*")):
                if file.is_file():
                    bundle.write(file, file.relative_to(checkpoint))
        partial.replace(archive)
        exported[checkpoint.name] = archive
        print(f"Saved checkpoint {archive}", flush=True)

    with subprocess.Popen(command) as process:
        try:
            while True:
                try:
                    return_code = process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    export_latest()
                    continue
                export_latest()
                if return_code:
                    raise subprocess.CalledProcessError(return_code, command)
                break
        finally:
            if process.poll() is None:
                process.terminate()

    last = (output / "checkpoints" / "last").resolve(strict=True)
    return exported[last.name]

smoke_checkpoint = train_act("smoke-l4", steps=20, save_freq=20)

## Train the baseline
Run this cell after the smoke test. It starts a fresh 20,000-step run with the same batch size and camera inputs. Every 5,000 steps it computes loss on the held-out episodes and copies the checkpoint, normalization processors, and training state to Drive.

Use offline loss to diagnose fitting. To select a checkpoint, compare actual lift and placement on the same development scenes in MuJoCo, then report results on separate test scenes. Keep the camera layout and control rate fixed across those runs.

In [ ]:
baseline_checkpoint = train_act("single-cube-act-l4", steps=20_000, save_freq=5_000)